# 01 · Red de quebradas: de Chosica a Chaclacayo

Pregunta: si una quebrada sobre Chosica se activa, ¿qué tramos del Rímac quedan aguas abajo, a cuántos kilómetros, y cuánto riesgo llega?

Modelo mínimo: los ríos y quebradas de OpenStreetMap como **grafo dirigido** (OSM dibuja cada cauce en el sentido del flujo) y un riesgo que baja por la corriente y se atenúa con la distancia. Es un ejercicio de exploración, **no** un sistema de alerta.

Datos: `uv run descargar-datos` · © colaboradores de OpenStreetMap (ODbL) · Weather data by Open-Meteo.com (CC BY 4.0).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from illapa_research.grafo import (
    aguas_abajo,
    cargar_osm,
    nodo_mas_cercano,
    propagar_riesgo,
)

DATOS = Path("../datos")
g = cargar_osm(DATOS / "osm_aguas_chosica.json")
print(f"{g.number_of_nodes()} nodos, {g.number_of_edges()} tramos, "
      f"{nx.number_weakly_connected_components(g)} redes separadas, "
      f"sin ciclos: {nx.is_directed_acyclic_graph(g)}")

## Comprobar el sentido del flujo

El Rímac baja de este a oeste. Si el grafo está bien orientado, el punto más al este del Rímac debe tener aguas abajo a todo el cauce, y el más al oeste, nada.

In [ ]:
nombres = nx.get_edge_attributes(g, "nombre")
rimac = {n for (a, b), nom in nombres.items() if nom == "Río Rímac" for n in (a, b)}
este = max(rimac, key=lambda n: g.nodes[n]["lon"])
oeste = min(rimac, key=lambda n: g.nodes[n]["lon"])
print("aguas abajo del extremo este :", len(aguas_abajo(g, este)) + 1, "de", len(rimac), "nodos del Rímac")
print("aguas abajo del extremo oeste:", len(aguas_abajo(g, oeste)))

## Propagar el riesgo desde una quebrada

Ponemos riesgo 1 en una quebrada sobre Chosica (el punto de la red más cercano a las coordenadas de abajo) y lo dejamos bajar con un decaimiento de 10 km.

In [ ]:
ORIGEN = (-76.665, -11.930)  # lon, lat — cámbialo por cualquier punto del mapa
origen = nodo_mas_cercano(g, *ORIGEN)
riesgo = propagar_riesgo(g, {origen: 1.0}, decaimiento_km=10)
alcanzados = [n for n, v in riesgo.items() if v > 0]
print(f"origen: nodo {origen} ({g.nodes[origen]['lon']:.4f}, {g.nodes[origen]['lat']:.4f})")
print(f"nodos aguas abajo: {len(alcanzados)}")

for nombre, lon in {"Chosica (origen)": ORIGEN[0], "Chaclacayo": -76.77}.items():
    cerca = [n for n in alcanzados if abs(g.nodes[n]["lon"] - lon) < 0.004]
    if cerca:
        print(f"{nombre}: riesgo máximo en el cauce ≈ {max(riesgo[n] for n in cerca):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for a, b in g.edges:
    ax.plot([g.nodes[a]["lon"], g.nodes[b]["lon"]], [g.nodes[a]["lat"], g.nodes[b]["lat"]],
            color="#c9ced6", lw=0.6, zorder=1)
nodos = list(alcanzados)
sc = ax.scatter([g.nodes[n]["lon"] for n in nodos], [g.nodes[n]["lat"] for n in nodos],
                c=[riesgo[n] for n in nodos], cmap="YlOrRd", s=6, zorder=2, vmin=0, vmax=1)
ax.scatter(g.nodes[origen]["lon"], g.nodes[origen]["lat"], marker="*", s=180, color="black", zorder=3, label="origen")
plt.colorbar(sc, label="riesgo relativo")
ax.set_aspect(1 / np.cos(np.radians(11.93)))
ax.set(title="Riesgo propagado aguas abajo (© OpenStreetMap)", xlabel="longitud", ylabel="latitud")
ax.legend(); plt.show()

## Lluvia diaria, 2015 a hoy

Los días extremos son los que nos importan. Tomamos como evento el 99.º percentil de los días con lluvia en Chosica.

In [ ]:
llu = json.loads((DATOS / "open_meteo_lluvia_diaria.json").read_text())
chosica = llu["chosica"]["daily"]
dias = np.array(chosica["time"], dtype="datetime64[D]")
mm = np.array([np.nan if v is None else v for v in chosica["precipitation_sum"]])
con_lluvia = mm[mm > 0.1]
umbral = np.nanpercentile(con_lluvia, 99)
print(f"{len(con_lluvia)} días con lluvia; umbral p99 = {umbral:.1f} mm/día")
print("días sobre el umbral:", [(str(d), float(v)) for d, v in zip(dias, mm) if v >= umbral][:10])

fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(dias, np.nan_to_num(mm), width=1, color="#2a6fb0")
ax.axhline(umbral, color="crimson", ls="--", label=f"p99 = {umbral:.1f} mm")
ax.set(title="Chosica · lluvia diaria (Open-Meteo)", ylabel="mm/día"); ax.legend(); plt.show()

## Siguientes pasos

- Orientar y completar los tramos que OSM no tenga (por ejemplo, con un modelo de elevación con licencia revisada).
- Cruzar con los distritos y la población (el INEI aún no se usa; el SENAMHI ver capítulo 11).
- Comparar con eventos pasados (los huaicos de marzo de 2015 en Chosica) antes de afirmar nada.